# Helpers for Testing

### Export archive as JSON

In [ ]:
import json

# Expected format of the archive:
# {(buildsite1, buildsite2, ...) : [reduction, coverage, fairness, -len(sites)]}

archive_to_save = archive     # swap `archive` for the name of your archive
output_path = 'archive.json'  # swap `archive.json` for the name of your output file

data = []
for sites, metrics in archive_to_save.items():
    reduction, coverage, fairness, _ = metrics
    data.append({
        'sol':       [int(s) for s in sites],
        'reduction': float(reduction),
        'coverage':  float(coverage),
        'fairness':  float(fairness),
    })

with open(output_path, 'w') as f:
    json.dump(data, f, indent=2)

print(f'saved {len(data)} solutions to {output_path}')

### Check archive for true non dominance

In [ ]:
import json
from pathlib import Path


def check_non_dominated(path):
    """Verify that every solution in the JSON archive at `path` is non-dominated.

    All 4 objectives are treated as bigger-is-better
    (reduction, coverage, fairness, -n_facilities).
    Facility count is derived from len(sol).
    If dominated entries are found, writes a cleaned copy alongside the input
    with `_true` appended to the filename.
    Returns True if the archive is a valid Pareto front.
    """
    path = Path(path)
    with open(path) as f:
        data = json.load(f)

    n = len(data)
    sols = [entry['sol'] for entry in data]
    objs = [
        (entry['reduction'], entry['coverage'], entry['fairness'], -len(entry['sol']))
        for entry in data
    ]

    def dominates(a, b):
        return all(x >= y for x, y in zip(a, b)) and any(x > y for x, y in zip(a, b))

    is_dominated = [False] * n
    dominated_by = {}
    for i in range(n):
        for j in range(n):
            if i != j and dominates(objs[j], objs[i]):
                is_dominated[i] = True
                dominated_by[i] = j
                break

    seen = {}
    duplicates = []
    for i, obj in enumerate(objs):
        if obj in seen:
            duplicates.append((seen[obj], i))
        else:
            seen[obj] = i

    survivors = [i for i in range(n) if not is_dominated[i]]

    print(f'checked {n} solutions from {path}')
    print(f'  non-dominated:         {len(survivors)}')
    print(f'  dominated:             {sum(is_dominated)}')
    print(f'  duplicate obj vectors: {len(duplicates)}')

    print(f'\n=== non-dominated solutions ({len(survivors)}) ===')
    print(f'{"n_fac":>5} {"reduction":>12} {"coverage":>12} {"fairness":>12}   sites')
    print('-' * 70)
    survivors_sorted = sorted(survivors, key=lambda i: (-(-objs[i][3]), -objs[i][0]))
    for i in survivors_sorted:
        reduction, coverage, fairness, neg_n = objs[i]
        print(f'{-int(neg_n):>5} {reduction:>12.2f} {coverage:>12.2f} {fairness:>12.2f}   {sols[i]}')

    if any(is_dominated):
        print(f'\n=== dominated entries ===')
        for i, j in list(dominated_by.items())[:20]:
            print(f'  [{i}] {objs[i]}  <-  dominated by [{j}] {objs[j]}')
        if sum(is_dominated) > 20:
            print(f'  ...and {sum(is_dominated) - 20} more')

    if duplicates:
        print(f'\n=== duplicate obj vectors (valid but redundant) ===')
        for i, j in duplicates[:10]:
            print(f'  [{i}] and [{j}]: {objs[i]}')

    is_clean = sum(is_dominated) == 0

    if not is_clean:
        cleaned = [
            {
                'sol':       sols[i],
                'reduction': objs[i][0],
                'coverage':  objs[i][1],
                'fairness':  objs[i][2],
            }
            for i in survivors_sorted
        ]
        cleaned_path = path.with_stem(path.stem + '_true')
        with open(cleaned_path, 'w') as f:
            json.dump(cleaned, f, indent=2)
        print(f'\narchive had dominated entries; cleaned {len(cleaned)} survivors saved to {cleaned_path}')
    else:
        print(f'\narchive is a valid Pareto front, no cleaned copy written')

    return is_clean


check_non_dominated('archive.json')

### Hypervolume on JSON

In [ ]:
# Bash command to install pymoo
# pip install pymoo

In [ ]:
import json
import numpy as np
from pymoo.indicators.hv import HV


def hypervolume(path, ref_point=np.array([0.0, 0.0, 0.0, 5.0])):
    """4D hypervolume for the archive at `path`.

    Fixed reference point [0, 0, 0, 5]:
      - axes 0/1/2 (reduction, coverage, fairness): worst = 0
        (all bigger-is-better, so negated values are <= 0)
      - axis 3 (facility count): worst = 5, one past the max of 4

    Every archive must be scored with the same ref_point for the numbers
    to be comparable across runs / people.
    """
    with open(path) as f:
        data = json.load(f)

    F = np.array([
        [entry['reduction'], entry['coverage'], entry['fairness'], -len(entry['sol'])]
        for entry in data
    ])
    # F columns: reduction, coverage, fairness, -n_facilities

    # pymoo minimises, so negate every axis
    hv = HV(ref_point=ref_point)(-F)

    print(f'file:        {path}')
    print(f'solutions:   {len(F)}')
    print(f'k range:     {-int(F[:, 3].max())} to {-int(F[:, 3].min())} facilities')
    print(f'ref point:   {ref_point.tolist()}')
    print(f'hypervolume: {hv:.4e}')

    return hv

# hypervolume('my_archive.json')  # swap for the name of your archive

file:        /Users/mia/Desktop/Solve4X/msf_workshop/my_archive.json
solutions:   43
k range:     2 to 4 facilities
ref point:   [0.0, 0.0, 0.0, 5.0]
hypervolume: 6.3803e+12


6380348634387.176